# Load fridge data

## 1.0a. Setup / Configs / vars

In [ ]:
PROJECT_ID= ! gcloud config list --format "value(core.project)" 2>/dev/null
PROJECT_ID=PROJECT_ID[0]
PROJECT_NBR= ! gcloud projects describe $PROJECT_ID | grep projectNumber | cut -d':' -f2 | xargs
PROJECT_NBR=PROJECT_NBR[0]
PROJECT_NAME= ! gcloud projects describe $PROJECT_ID | grep name | cut -d':' -f2 | xargs
PROJECT_NAME=PROJECT_NAME[0]
FRIDGE_STORAGE_BUCKET=f"rscw-workshop-fridge-stage-{PROJECT_NBR}"
LOCATION="us-central1"

## 1.0b. Create BQ dataset

## 2.0. Helper functions

In [ ]:
def RunQuery(sql):
  import time
  from google.cloud import bigquery
  client = bigquery.Client(location=f"{LOCATION}")

  if (sql.startswith("SELECT") or sql.startswith("WITH")):
      df_result = client.query(sql).to_dataframe()
      return df_result
  else:
    job_config = bigquery.QueryJobConfig(priority=bigquery.QueryPriority.INTERACTIVE)
    query_job = client.query(sql, job_config=job_config)

    # Check on the progress by getting the job's updated state.
    query_job = client.get_job(
        query_job.job_id, location=query_job.location
    )
    print("Job {} is currently in state {} with error result of {}".format(query_job.job_id, query_job.state, query_job.error_result))

    while query_job.state != "DONE":
      time.sleep(2)
      query_job = client.get_job(
          query_job.job_id, location=query_job.location
          )
      print("Job {} is currently in state {} with error result of {}".format(query_job.job_id, query_job.state, query_job.error_result))

    if query_job.error_result == None:
      return True
    else:
      raise Exception(query_job.error_result)

## 3. Create and load product_master and product_docs_ref_data tables

### 3.1. Create tables

Run the below to create the bigquery tables:

In [ ]:
%%bigquery --project {PROJECT_ID} --location {LOCATION}
--- DDL for table: customer_master
CREATE OR REPLACE TABLE `rscw_fridge_ds.product_master` (
  item_number STRING,
  appliance_type STRING,
  appliance_sub_type STRING,
  brand STRING,
  model_id STRING,
  omni_item_id STRING,
  description STRING,
  price NUMERIC,
  product_image_gcs_uri STRING,
  is_active STRING);



CREATE OR REPLACE TABLE `rscw_fridge_ds.product_docs_ref_data` (
  item_number STRING ,
  brand STRING,
  model_id STRING,
  pdf_name STRING,
  product_doc_type STRING,
  product_doc_gcs_uri STRING, );


Query is running:   0%|          |

""


### 3.2. Load product master data

In [ ]:
fridge_product_master_load_sql = f"""
LOAD DATA OVERWRITE rscw_fridge_ds.product_master(
  item_number STRING,
  appliance_type STRING,
  appliance_sub_type STRING,
  brand STRING,
  model_id STRING,
  omni_item_id STRING,
  description STRING,
  price NUMERIC,
  product_image_gcs_uri STRING,
  is_active STRING)
FROM FILES(format = 'CSV', uris = ARRAY['gs://{FRIDGE_STORAGE_BUCKET}/product_master.csv'],
  field_delimiter = ',', skip_leading_rows=1);

"""

In [ ]:
RunQuery(fridge_product_master_load_sql)

Job 186825b7-0ba4-43eb-bb9f-5c54edc74f0a is currently in state RUNNING with error result of None
Job 186825b7-0ba4-43eb-bb9f-5c54edc74f0a is currently in state DONE with error result of None


True

In [ ]:
%%bigquery --project {PROJECT_ID} --location {LOCATION}

--Update the GCS URI for product images
UPDATE
  rscw_fridge_ds.product_master
SET
  -- Set the column to its new value
  product_image_gcs_uri = REPLACE(product_image_gcs_uri, 'PROJECT_NUMBER', '{PROJECT_NBR}'),
  brand=UPPER(brand)
WHERE 1=1;



Query is running:   0%|          |

""


### 3.3. Load product docs reference data

In [ ]:
fridge_product_docs_load_sql = f"""
LOAD DATA OVERWRITE rscw_fridge_ds.product_docs_ref_data(
  item_number STRING,
  brand STRING,
  model_id STRING,
  pdf_name STRING,
  product_doc_type STRING,
  product_doc_gcs_uri STRING)
FROM FILES(format = 'CSV', uris = ARRAY['gs://{FRIDGE_STORAGE_BUCKET}/product_doc_ref_data.csv'],
  field_delimiter = ',', skip_leading_rows=1);
"""

In [ ]:
RunQuery(fridge_product_docs_load_sql)

Job bad70103-3084-4bd5-a8e2-8d7f57fe3fd9 is currently in state RUNNING with error result of None
Job bad70103-3084-4bd5-a8e2-8d7f57fe3fd9 is currently in state DONE with error result of None


True

In [ ]:
fridge_product_docs_update_sql = f"""


UPDATE
  rscw_fridge_ds.product_docs_ref_data
SET
  -- Set the column to its new value
  product_doc_gcs_uri = REPLACE(product_doc_gcs_uri, 'PROJECT_NUMBER', '{PROJECT_NBR}'),
  brand=UPPER(brand)
-- Optional: A WHERE clause can be added to filter which rows are affected
WHERE 1=1;

"""

In [ ]:
RunQuery(fridge_product_docs_update_sql)

Job e45a227b-568e-4eb0-9851-c7c741ef3ff7 is currently in state RUNNING with error result of None
Job e45a227b-568e-4eb0-9851-c7c741ef3ff7 is currently in state DONE with error result of None


True

## 4. Create and load supplier master and product supplier data

### 4.1. Create tables

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}
--- DDL for table: customer_master
CREATE OR REPLACE TABLE `rscw_fridge_ds.supplier_master` (
  supplier_id STRING,
  supplier_name STRING,
  contact_name STRING,
  contact_email STRING,
  address STRING,
  city STRING,
  state_code STRING,
  zip_cd STRING,
  country_code STRING,
  phone_number STRING
);



CREATE OR REPLACE TABLE `rscw_fridge_ds.product_suppliers`
(
  item_number STRING,
  omni_item_id STRING,
  supplier_id STRING,
  supplier_type STRING,
  lead_days_to_delivery INTEGER
);

Query is running:   0%|          |

""


### 4.2. Load supplier_master

In [ ]:
supplier_master_load_sql = f"""
LOAD DATA OVERWRITE rscw_fridge_ds.supplier_master(
  supplier_id STRING,
  supplier_name STRING,
  contact_name STRING,
  contact_email STRING,
  address STRING,
  city STRING,
  state_code STRING,
  zip_cd STRING,
  country_code STRING,
  phone_number STRING)
FROM FILES(format = 'CSV', uris = ARRAY['gs://{FRIDGE_STORAGE_BUCKET}/supplier_master.csv'],
  field_delimiter = ',', skip_leading_rows=1);
"""

In [ ]:
RunQuery(supplier_master_load_sql)

Job fd979b92-7a09-46a5-8f30-51b1cf74f791 is currently in state RUNNING with error result of None
Job fd979b92-7a09-46a5-8f30-51b1cf74f791 is currently in state DONE with error result of None


True

### 4.3. Load product_suppliers

In [ ]:
product_suppliers_load_sql = f"""
delete from rscw_fridge_ds.product_suppliers
where 1=1;

insert into rscw_fridge_ds.product_suppliers(item_number, omni_item_id,supplier_id,supplier_type,lead_days_to_delivery)
select item_number,omni_item_id,UPPER(brand),'primary',7
from rscw_fridge_ds.product_master;
"""

In [ ]:
RunQuery(product_suppliers_load_sql)

Job 231a4dd1-9735-4acd-a39b-37196f5f3c47 is currently in state RUNNING with error result of None
Job 231a4dd1-9735-4acd-a39b-37196f5f3c47 is currently in state RUNNING with error result of None
Job 231a4dd1-9735-4acd-a39b-37196f5f3c47 is currently in state DONE with error result of None


True

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

update rscw_fridge_ds.product_suppliers
set supplier_id=trim(supplier_id) where 1=1;

update rscw_fridge_ds.supplier_master
set supplier_id=trim(supplier_id) where 1=1;

Query is running:   0%|          |

""


## 5. Load customer master

### 5.1. Create table

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE `rscw_fridge_ds.customer_master`
 (customer_id STRING ,
  first_name STRING ,
  last_name STRING ,
  email STRING ,
  address STRING ,
  city STRING ,
  state_code STRING ,
  zip_code STRING ,
  country_code STRING ,
  phone_number STRING);


Query is running:   0%|          |

""


### 5.2. Load table

In [ ]:
customer_master_load_sql = f"""
LOAD DATA OVERWRITE rscw_fridge_ds.customer_master(
  customer_id STRING ,
  first_name STRING ,
  last_name STRING ,
  email STRING ,
  address STRING ,
  city STRING ,
  state_code STRING ,
  zip_code STRING ,
  country_code STRING ,
  phone_number STRING)
FROM FILES(format = 'CSV', uris = ARRAY['gs://{FRIDGE_STORAGE_BUCKET}/customer_master.csv'],
  field_delimiter = '|');
"""

RunQuery(customer_master_load_sql)

Job b8769f5f-dbbd-40ee-b760-344d29a4da1b is currently in state RUNNING with error result of None
Job b8769f5f-dbbd-40ee-b760-344d29a4da1b is currently in state RUNNING with error result of None
Job b8769f5f-dbbd-40ee-b760-344d29a4da1b is currently in state DONE with error result of None


True

## 6. Create and load fleet, location, driver

### 6.1. Create tables

In [ ]:

%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE `rscw_fridge_ds.driver_master` (driver_id STRING ,    employee_id STRING );

CREATE OR REPLACE TABLE `rscw_fridge_ds.fleet_master` (vehicle_id STRING ,    license_plate STRING ,    vin STRING ,    model STRING ,    status STRING );

CREATE OR REPLACE TABLE `rscw_fridge_ds.location_master` (location_id STRING ,    location_name STRING ,    location_type STRING ,    address STRING ,    city STRING ,    state_code STRING ,    zip_code STRING ,    country_code STRING ,    phone_number STRING );


Query is running:   0%|          |

""


### 6.2. Load driver master

In [ ]:
driver_master_load_sql = f"""
LOAD DATA OVERWRITE rscw_fridge_ds.driver_master(
  driver_id STRING ,    employee_id STRING )
FROM FILES(format = 'CSV', uris = ARRAY['gs://{FRIDGE_STORAGE_BUCKET}/driver_master.csv'],
  field_delimiter = '|');
"""

RunQuery(driver_master_load_sql)

Job 49fa5694-3621-4498-9f3b-814efebb4f29 is currently in state RUNNING with error result of None
Job 49fa5694-3621-4498-9f3b-814efebb4f29 is currently in state DONE with error result of None


True

### 6.3. Load fleet master

In [ ]:
fleet_master_load_sql = f"""
LOAD DATA OVERWRITE rscw_fridge_ds.fleet_master(
  vehicle_id STRING ,    license_plate STRING ,    vin STRING ,    model STRING ,    status STRING )
FROM FILES(format = 'CSV', uris = ARRAY['gs://{FRIDGE_STORAGE_BUCKET}/fleet_master.csv'],
  field_delimiter = '|');
"""

RunQuery(fleet_master_load_sql)

Job e88fcb10-8567-4174-886c-7752a146003c is currently in state RUNNING with error result of None
Job e88fcb10-8567-4174-886c-7752a146003c is currently in state DONE with error result of None


True

### 6.4. Load location master

In [ ]:
location_master_load_sql = f"""
LOAD DATA OVERWRITE rscw_fridge_ds.location_master(
  location_id STRING ,    location_name STRING ,    location_type STRING ,    address STRING ,    city STRING ,    state_code STRING ,    zip_code STRING ,    country_code STRING ,    phone_number STRING )
FROM FILES(format = 'CSV', uris = ARRAY['gs://{FRIDGE_STORAGE_BUCKET}/location_master.csv'],
  field_delimiter = '|');
"""

RunQuery(location_master_load_sql)

Job 3ef2e574-3cd6-4302-9948-0baa82001e7c is currently in state RUNNING with error result of None
Job 3ef2e574-3cd6-4302-9948-0baa82001e7c is currently in state DONE with error result of None


True

## 7. Purchase orders and purchase order items

### 7.1. Purchase orders

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE `rscw_fridge_ds.purchase_orders` (
  po_id STRING ,
  supplier_id STRING,
  order_date DATE,
  expected_date DATE,
  received_date DATE,
  total_cost DECIMAL,
  order_status  STRING);


Query is running:   0%|          |

""


In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

DELETE FROM `rscw_fridge_ds.purchase_orders` WHERE 1=1;

INSERT INTO `rscw_fridge_ds.purchase_orders` (po_id,supplier_id,order_date,expected_date,received_date,total_cost,order_status)
select GENERATE_UUID(),'SAMSUNG',CAST('2024-12-23' AS DATE),CAST('2024-12-30' AS DATE),CAST('2024-12-30' AS DATE),0,'FULFILLED';
INSERT INTO `rscw_fridge_ds.purchase_orders` (po_id,supplier_id,order_date,expected_date,received_date,total_cost,order_status)
select GENERATE_UUID(),'FRIGIDAIRE',CAST('2024-12-23' AS DATE),CAST('2024-12-30' AS DATE),CAST('2024-12-30' AS DATE),0,'FULFILLED';
INSERT INTO `rscw_fridge_ds.purchase_orders` (po_id,supplier_id,order_date,expected_date,received_date,total_cost,order_status)
select GENERATE_UUID(),'LG',CAST('2024-12-23' AS DATE),CAST('2024-12-30' AS DATE),CAST('2024-12-30' AS DATE),0,'FULFILLED';
INSERT INTO `rscw_fridge_ds.purchase_orders` (po_id,supplier_id,order_date,expected_date,received_date,total_cost,order_status)
select GENERATE_UUID(),'WHIRLPOOL',CAST('2024-12-23' AS DATE),CAST('2024-12-30' AS DATE),CAST('2024-12-30' AS DATE),0,'FULFILLED';

Query is running:   0%|          |

""


### 7.2. Purchase order items

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE `rscw_fridge_ds.purchase_order_items` (
  po_id STRING ,
  line_item_id STRING,
  item_number STRING,
  omni_item_id STRING,
  quantity_ordered  INTEGER,
  unit_price DECIMAL);

Query is running:   0%|          |

""


In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

DELETE FROM `rscw_fridge_ds.purchase_order_items` WHERE 1=1;

INSERT INTO `rscw_fridge_ds.purchase_order_items` (po_id,line_item_id,item_number,omni_item_id,quantity_ordered,unit_price)
select po.po_id,GENERATE_UUID(),pm.item_number, pm.omni_item_id, 600,pm.price
from rscw_fridge_ds.purchase_orders po
join rscw_fridge_ds.product_master pm
on po.supplier_id=pm.brand
where pm.is_active='Y';

Query is running:   0%|          |

""


### 7.3. Update purchase order table with total cost

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

MERGE rscw_fridge_ds.purchase_orders PO
USING (
  SELECT po_id, SUM(quantity_ordered * unit_price) AS total_cost
  FROM rscw_fridge_ds.purchase_order_items
  GROUP BY po_id
) AS POI
ON PO.po_id = POI.po_id
WHEN MATCHED THEN
  UPDATE SET total_cost = POI.total_cost

Query is running:   0%|          |

""


## 8. Stock allocation across stores and warehouse

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE `rscw_fridge_ds.stock_allocation_plan` (
  allocation_date DATE ,
  item_number STRING,
  omni_item_id STRING,
  location_id STRING,
  quantity INTEGER,
  update_date DATE);

Query is running:   0%|          |

""


In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

DELETE FROM rscw_fridge_ds.stock_allocation_plan
WHERE 1=1;

INSERT INTO rscw_fridge_ds.stock_allocation_plan(allocation_date,item_number,omni_item_id,location_id,quantity,update_date)
SELECT CAST('2024-12-31' as DATE),pm.item_number,pm.omni_item_id,'NAP-IL-ST',40,CAST('2024-12-31' as DATE)
FROM rscw_fridge_ds.product_master pm where is_active='Y';

INSERT INTO rscw_fridge_ds.stock_allocation_plan(allocation_date,item_number,omni_item_id,location_id,quantity,update_date)
SELECT CAST('2024-12-31' as DATE),pm.item_number,pm.omni_item_id,'SCH-IL-ST',40,CAST('2024-12-31' as DATE)
FROM rscw_fridge_ds.product_master pm where is_active='Y';

INSERT INTO rscw_fridge_ds.stock_allocation_plan(allocation_date,item_number,omni_item_id,location_id,quantity,update_date)
SELECT CAST('2024-12-31' as DATE),pm.item_number,pm.omni_item_id,'CHI-IL-ST',40,CAST('2024-12-31' as DATE)
FROM rscw_fridge_ds.product_master pm where is_active='Y';

INSERT INTO rscw_fridge_ds.stock_allocation_plan(allocation_date,item_number,omni_item_id,location_id,quantity,update_date)
SELECT CAST('2024-12-31' as DATE),pm.item_number,pm.omni_item_id,'WHE-IL-WH',480,CAST('2024-12-31' as DATE)
FROM rscw_fridge_ds.product_master pm where is_active='Y';

Query is running:   0%|          |

""


## 9. Stock movement

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE `rscw_fridge_ds.stock_movement` (
  movement_date DATE ,
  item_number STRING,
  omni_item_id STRING,
  movement_type STRING, --INITIAL_STOCK, RESTOCK, SALE, RETURN, WAREHOUSE_TO_STORE, STORE_TO_WAREHOUSE, STORE_TO_STORE, SUPPLIER_DELIVERY
  location_id STRING,
  quantity_change INTEGER,
  reference_id STRING,
  reference_id_type STRING);


Query is running:   0%|          |

""


In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}


DELETE FROM rscw_fridge_ds.stock_movement
WHERE 1=1;

INSERT INTO rscw_fridge_ds.stock_movement(movement_date,item_number,omni_item_id,movement_type,location_id,quantity_change,reference_id,reference_id_type)
SELECT CAST('2024-12-31' as DATE),poi.item_number,poi.omni_item_id,'INITIAL_STOCK','NAP-IL-ST',40,poi.po_id,'PURCHASE_ORDER'
FROM `rscw_fridge_ds.purchase_order_items` poi;

INSERT INTO rscw_fridge_ds.stock_movement(movement_date,item_number,omni_item_id,movement_type,location_id,quantity_change,reference_id,reference_id_type)
SELECT CAST('2024-12-31' as DATE),poi.item_number,poi.omni_item_id,'INITIAL_STOCK','SCH-IL-ST',40,poi.po_id,'PURCHASE_ORDER'
FROM `rscw_fridge_ds.purchase_order_items` poi;

INSERT INTO rscw_fridge_ds.stock_movement(movement_date,item_number,omni_item_id,movement_type,location_id,quantity_change,reference_id,reference_id_type)
SELECT CAST('2024-12-31' as DATE),poi.item_number,poi.omni_item_id,'INITIAL_STOCK','CHI-IL-ST',40,poi.po_id,'PURCHASE_ORDER'
FROM `rscw_fridge_ds.purchase_order_items` poi;

INSERT INTO rscw_fridge_ds.stock_movement(movement_date,item_number,omni_item_id,movement_type,location_id,quantity_change,reference_id,reference_id_type)
SELECT CAST('2024-12-31' as DATE),poi.item_number,poi.omni_item_id,'SUPPLIER_DELIVERY','WHE-IL-WH',600,poi.po_id,'PURCHASE_ORDER'
FROM `rscw_fridge_ds.purchase_order_items` poi;

INSERT INTO rscw_fridge_ds.stock_movement(movement_date,item_number,omni_item_id,movement_type,location_id,quantity_change,reference_id,reference_id_type)
SELECT CAST('2024-12-31' as DATE),poi.item_number,poi.omni_item_id,'WAREHOUSE_TO_STORE','WHE-IL-WH',-120,poi.po_id,'PURCHASE_ORDER'
FROM `rscw_fridge_ds.purchase_order_items` poi;

Query is running:   0%|          |

""


## 10. Stock transfer orders and Stock transfer order items

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE rscw_fridge_ds.stock_transfer_orders (
    stock_transfer_order_id STRING,
    from_location_id STRING,
    to_location_id STRING,
    status STRING,
    vehicle_id STRING,
    reference_id STRING,
    reference_id_type STRING,
    creation_date DATE,
    update_date DATE
);

Query is running:   0%|          |

""


In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

DELETE FROM rscw_fridge_ds.stock_transfer_orders WHERE 1=1;

-- Naperville store
INSERT INTO rscw_fridge_ds.stock_transfer_orders(stock_transfer_order_id,from_location_id,to_location_id,status,vehicle_id,reference_id,reference_id_type,creation_date,update_date)
SELECT GENERATE_UUID(),'WHE-IL-WH','NAP-IL-ST','DELIVERED','Nomad',PO.po_id,'PURCHASE_ORDER',CAST('2024-12-15' AS DATE),CAST('2024-12-31' as DATE)
FROM rscw_fridge_ds.purchase_orders PO;

-- Schaumburg store
INSERT INTO rscw_fridge_ds.stock_transfer_orders(stock_transfer_order_id,from_location_id,to_location_id,status,vehicle_id,reference_id,reference_id_type,creation_date,update_date)
SELECT GENERATE_UUID(),'WHE-IL-WH','SCH-IL-ST','DELIVERED','RustyRose',PO.po_id,'PURCHASE_ORDER',CAST('2024-12-15' AS DATE),CAST('2024-12-31' as DATE)
FROM rscw_fridge_ds.purchase_orders PO;


-- Chicago store
INSERT INTO rscw_fridge_ds.stock_transfer_orders(stock_transfer_order_id,from_location_id,to_location_id,status,vehicle_id,reference_id,reference_id_type,creation_date,update_date)
SELECT GENERATE_UUID(),'WHE-IL-WH','CHI-IL-ST','DELIVERED','Dart',PO.po_id,'PURCHASE_ORDER',CAST('2024-12-15' AS DATE),CAST('2024-12-31' as DATE)
FROM rscw_fridge_ds.purchase_orders PO;


Query is running:   0%|          |

""


In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE rscw_fridge_ds.stock_transfer_order_items (
    stock_transfer_order_id STRING,
    item_number STRING,
    omni_item_id STRING,
    quantity INT
);

Query is running:   0%|          |

""


In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

DELETE FROM rscw_fridge_ds.stock_transfer_order_items WHERE 1=1;

INSERT INTO rscw_fridge_ds.stock_transfer_order_items(stock_transfer_order_id,item_number,omni_item_id,quantity)
SELECT STO.stock_transfer_order_id,POI.item_number,POI.omni_item_id,40
FROM rscw_fridge_ds.stock_transfer_orders STO JOIN rscw_fridge_ds.purchase_orders PO on (STO.reference_id = PO.po_id)
JOIN rscw_fridge_ds.purchase_order_items POI on (PO.po_id = POI.po_id);


Query is running:   0%|          |

""


## 11. Stock thresholds

In [ ]:

%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE rscw_fridge_ds.stock_thresholds (
item_number	STRING,
omni_item_id STRING,
stock_on_hand_across_stores	INT,
stock_at_each_store	INT,
total_stock_at_stores	INT,
total_stock_at_warehouse	INT,
average_sold_per_day_per_store	INT,
avg_sold_per_day_total	INT,
safety_stock_per_store	INT,
safety_stock_total	INT,
reorder_point_per_store	INT,
reorder_point_total	INT,
last_updated_date	DATE,
is_current STRING);

Query is running:   0%|          |

""


In [ ]:
stock_thresholds_load_sql = f"""
LOAD DATA OVERWRITE rscw_fridge_ds.stock_thresholds(
item_number	STRING,
omni_item_id STRING,
stock_on_hand_across_stores	INT,
stock_at_each_store	INT,
total_stock_at_stores	INT,
total_stock_at_warehouse	INT,
average_sold_per_day_per_store	INT,
avg_sold_per_day_total	INT,
safety_stock_per_store	INT,
safety_stock_total	INT,
reorder_point_per_store	INT,
reorder_point_total	INT,
last_updated_date	DATE,
is_current STRING)
FROM FILES(format = 'CSV', uris = ARRAY['gs://{FRIDGE_STORAGE_BUCKET}/stock_thresholds.csv'],
  field_delimiter = ',', skip_leading_rows=1);
"""

RunQuery(stock_thresholds_load_sql)

Job f7deda4c-b930-4d9c-865b-bd3d48511b31 is currently in state RUNNING with error result of None
Job f7deda4c-b930-4d9c-865b-bd3d48511b31 is currently in state DONE with error result of None


True

## 12. Stock master table

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE rscw_fridge_ds.stock_master (
    stock_date DATE,
    item_number STRING,
    omni_item_id STRING,
    quantity_on_hand INT,
    safety_stock INT,
    reorder_point INT
);


CREATE OR REPLACE TABLE rscw_fridge_ds.stock_master_location (
    stock_date DATE,
    item_number STRING,
    omni_item_id STRING,
    location_id STRING,
    quantity_on_hand INT
);

Query is running:   0%|          |

""


In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

DELETE FROM rscw_fridge_ds.stock_master_location WHERE 1=1;

insert into rscw_fridge_ds.stock_master_location(stock_date,item_number,omni_item_id,location_id,quantity_on_hand)
select movement_date, item_number, omni_item_id, location_id, sum(quantity_change) stock from rscw_fridge_ds.stock_movement
group by movement_date, item_number, omni_item_id, location_id
order by movement_date, item_number, omni_item_id, location_id;

Query is running:   0%|          |

""


In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}

DELETE FROM rscw_fridge_ds.stock_master WHERE 1=1;

insert into rscw_fridge_ds.stock_master(stock_date,item_number,omni_item_id,quantity_on_hand,safety_stock,reorder_point)
select SLM.stock_date, SLM.item_number, SLM.omni_item_id, sum(SLM.quantity_on_hand), ST.safety_stock_total, ST.reorder_point_total
from rscw_fridge_ds.stock_master_location SLM
join rscw_fridge_ds.stock_thresholds ST on (SLM.item_number = ST.item_number and SLM.omni_item_id=ST.omni_item_id)
group by SLM.stock_date, SLM.item_number, SLM.omni_item_id,ST.safety_stock_total,ST.reorder_point_total
order by SLM.stock_date, item_number, omni_item_id;



Query is running:   0%|          |

""


## 13. POS transactions

### 13.1. Create tables

In [ ]:
%%bigquery  --project {PROJECT_ID} --location {LOCATION}


--- DDL for table: pos_transaction_items
CREATE OR REPLACE TABLE `rscw_fridge_ds.pos_transaction_items` (
  transaction_id STRING,
  item_number STRING,
  omni_item_id STRING,
  quantity INT,
  price DECIMAL,
  line_item_total DECIMAL);


--- DDL for table: pos_transactions
CREATE OR REPLACE TABLE `rscw_fridge_ds.pos_transactions` (
  transaction_id STRING,
  location_id STRING,
  customer_id STRING,
  transaction_status STRING,
  transaction_date DATE,
  payment_type STRING,
  payment_total_dollar DECIMAL);


Query is running:   0%|          |

""


### 13.2. Generate POS transactions

In [ ]:
RunQuery("delete from rscw_fridge_ds.pos_transactions where 1=1")
RunQuery("delete from rscw_fridge_ds.pos_transaction_items where 1=1")

Job 83d29aed-c173-4f32-8bf5-8f293c1631cf is currently in state RUNNING with error result of None
Job 83d29aed-c173-4f32-8bf5-8f293c1631cf is currently in state DONE with error result of None
Job d63fc045-5c31-4b29-8dab-75971b713965 is currently in state RUNNING with error result of None
Job d63fc045-5c31-4b29-8dab-75971b713965 is currently in state DONE with error result of None


True

In [ ]:
import pandas as pd
import uuid
import random
from datetime import datetime, timedelta
from google.cloud import bigquery

# 1. INITIALIZATION & BIGQUERY SETUP
client = bigquery.Client()
project_id = PROJECT_ID # Replace with your GCP Project ID
dataset_id = "rscw_fridge_ds"
locations = ['NAP-IL-ST', 'SCH-IL-ST', 'CHI-IL-ST']

# 2. FETCH MASTER DATA (Product Specs & Thresholds)
# This query joins your masters to get the baseline sales target for every SKU
products_query = f"""
    SELECT
        p.item_number,
        p.omni_item_id,
        p.price,
        t.average_sold_per_day_per_store
    FROM `{project_id}.{dataset_id}.product_master` p
    JOIN `{project_id}.{dataset_id}.stock_thresholds` t
      ON p.item_number = t.item_number AND p.omni_item_id = t.omni_item_id
    WHERE p.is_active='Y'
"""
products_df = client.query(products_query).to_dataframe()

# 3. CUSTOMER POOL GENERATOR
# We fetch all unique customers and create a generator to ensure 1 fridge/customer ever
customer_query = f"SELECT customer_id FROM `{project_id}.{dataset_id}.customer_master`"
customer_list = client.query(customer_query).to_dataframe()['customer_id'].tolist()
random.shuffle(customer_list)
customer_pool = (c for c in customer_list)

def process_daily_ingestion(start_date,end_date):
    current_date = start_date
    total_days = (end_date - start_date).days + 1

    print(f"Starting Daily Ingestion for batch from dates {start_date} to {end_date} - a total of {total_days} days...")

    while current_date <= end_date:
        print(f"Generating data for  {current_date}...")
        # Containers for the current day's data
        daily_tx = []
        daily_tx_items = []

        # Loop through every Store
        for loc in locations:
            # Loop through every Product
            for _, product in products_df.iterrows():

                # RELEVANT SPECS:
                # Sell exactly 'average_sold_per_day_per_store' + random (0-10)
                # This guarantees that the average is ALWAYS met or exceeded.
                avg_val = int(product['average_sold_per_day_per_store'])
                units_to_sell = random.randint(avg_val, avg_val + 5)

                for _ in range(units_to_sell):
                    try:
                        # Ensures the one-fridge-per-customer rule
                        cust_id = next(customer_pool)
                    except StopIteration:
                        print("CRITICAL: Customer master exhausted. Ingestion stopping.")
                        return

                    trx_id = str(uuid.uuid4())
                    price = float(product['price'])

                    # Create Transaction Header
                    daily_tx.append({
                        "transaction_id": trx_id,
                        "location_id": loc,
                        "customer_id": cust_id,
                        "transaction_status": "SALE",
                        "transaction_date": current_date.date(),
                        "payment_type": random.choice(["CREDIT", "DEBIT", "CASH"]),
                        "payment_total_dollar": price
                    })

                    # Create Transaction Line Item
                    daily_tx_items.append({
                        "transaction_id": trx_id,
                        "item_number": product['item_number'],
                        "omni_item_id": product['omni_item_id'],
                        "quantity": 1,
                        "price": price,
                        "line_item_total": price
                    })

        # UPLOAD TO BIGQUERY (Daily Commit)
        if daily_tx:
            upload_day(daily_tx, "pos_transactions", current_date)
            upload_day(daily_tx_items, "pos_transaction_items", current_date)

        # Explicit memory cleanup before starting the next day
        del daily_tx
        del daily_tx_items

        current_date += timedelta(days=1)

def upload_day(data_list, table_name, day_date):
    """Helper to upload a single day of data via pandas-gbq"""
    df = pd.DataFrame(data_list)
    table_ref = f"{dataset_id}.{table_name}"

    # Using 'append' to continuously build the year's data
    df.to_gbq(table_ref, project_id=project_id, if_exists='append')

    # Console feedback for monitoring
    if "items" not in table_name:
        print(f"[{day_date.date()}] Inserted {len(df)} transactions.")



In [ ]:
print("Batch 1 start datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 3, 31)
process_daily_ingestion(start_date,end_date)
print("Batch 1 end datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

In [ ]:
# Batch 2
print("Batch 2 start datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
start_date = datetime(2025, 4, 1)
end_date = datetime(2025, 6, 30)
process_daily_ingestion(start_date,end_date)
print("Batch 2 end datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

In [ ]:
# Batch 3
print("Batch 3 start datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
start_date = datetime(2025, 7, 1)
end_date = datetime(2025, 9, 30)
process_daily_ingestion(start_date,end_date)
print("Batch 3 end datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

In [ ]:
# Batch 4

customer_query = f"SELECT customer_id FROM `{project_id}.{dataset_id}.customer_master`"
customer_list = client.query(customer_query).to_dataframe()['customer_id'].tolist()
random.shuffle(customer_list)
customer_pool = (c for c in customer_list)

print("Batch 4 start datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
start_date = datetime(2025, 10, 1)
end_date = datetime(2025, 11, 21)
process_daily_ingestion(start_date,end_date)
print("Batch 4 end datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

In [ ]:
# Batch 5
print("Batch 5 start datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
start_date = datetime(2025, 11, 22)
end_date = datetime(2026, 1, 31)
process_daily_ingestion(start_date,end_date)
print("Batch 5 end datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

## 14. Create the metadata dataset and tables to persist Data Insights scan results

In [ ]:
%%bigquery --project {PROJECT_ID}

CREATE SCHEMA rscw_fridge_metadata_ds OPTIONS (location = 'us-central1');

In [ ]:
%%bigquery --project {PROJECT_ID} --location {LOCATION}

CREATE OR REPLACE TABLE `rscw_fridge_metadata_ds.dataset_description`
(
  dataset_description STRING
);

CREATE OR REPLACE TABLE `rscw_fridge_metadata_ds.dataset_table_relationships`
(
  table_1 STRING,
  table_1_column STRING,
  table_2 STRING,
  table_2_column STRING,
  join_type STRING
);

CREATE OR REPLACE TABLE `rscw_fridge_metadata_ds.table_column_descriptions`
(
  table_name STRING,
  column_name STRING,
  column_description STRING
);

CREATE OR REPLACE TABLE `rscw_fridge_metadata_ds.table_descriptions`
(
  name STRING,
  description STRING
);